In [1]:
from langchain_community.document_loaders import PyPDFLoader

# PDF 파일을 읽어서 텍스트 데이터 호출
loader = PyPDFLoader("../data/OneNYC_2050_StrategicPlan.pdf")
data_nyc = loader.load()
print(data_nyc)

C:\Users\bny64\AppData\Local\Temp\ipykernel_10708\2716045293.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 14.0 (Windows)', 'creationdate': '2019-04-30T10:57:03-04:00', 'moddate': '2020-01-03T16:36:59-05:00', 'trapped': '/False', 'source': '../data/OneNYC_2050_StrategicPlan.pdf', 'total_pages': 60, 'page': 0, 'page_label': '1'}, page_content='OneNYC \n2050\nBUILDING A STRONG \nAND FAIR CITY \nVOLUME 1 OF 9  \nA\nPRIL 2019\nTHE CITY OF NEW YORK\nMAYOR BILL DE BLASIO\nDEAN FULEIHAN  \nFIRST DEPUTY MAYOR \nD\nOMINIC WILLIAMS  \nCHIEF POLICY ADVISOR\nD\nANIEL A. ZARRILLI  \nOne NYC DIRECTOR'), Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 14.0 (Windows)', 'creationdate': '2019-04-30T10:57:03-04:00', 'moddate': '2020-01-03T16:36:59-05:00', 'trapped': '/False', 'source': '../data/OneNYC_2050_StrategicPlan.pdf', 'total_pages': 60, 'page': 1, 'page_label': '2'}, page_content='2   |   O neNYC 2050 \nNYC.GOV/O neNYC\nONENYC 2050 IS A STRATEGY TO SECURE OUR CITY’S \nFUTURE AGAINST

In [2]:
# 긴 텍스트 데이터를 청크 단위로 나누는 모듈
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1000 단위씩 안전하게 100자의 오버랩을 통해 중요한 정보가 누락되지 않도록 함
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
all_splits = text_splitter.split_documents(data_nyc)

print(len(all_splits))
for i, split in enumerate(all_splits):
    print(f"Split {i+1}:------------------------------------\n")
    print(split)

176
Split 1:------------------------------------

page_content='OneNYC 
2050
BUILDING A STRONG 
AND FAIR CITY 
VOLUME 1 OF 9  
A
PRIL 2019
THE CITY OF NEW YORK
MAYOR BILL DE BLASIO
DEAN FULEIHAN  
FIRST DEPUTY MAYOR 
D
OMINIC WILLIAMS  
CHIEF POLICY ADVISOR
D
ANIEL A. ZARRILLI  
One NYC DIRECTOR' metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 14.0 (Windows)', 'creationdate': '2019-04-30T10:57:03-04:00', 'moddate': '2020-01-03T16:36:59-05:00', 'trapped': '/False', 'source': '../data/OneNYC_2050_StrategicPlan.pdf', 'total_pages': 60, 'page': 0, 'page_label': '1'}
Split 2:------------------------------------

page_content='2   |   O neNYC 2050 
NYC.GOV/O neNYC
ONENYC 2050 IS A STRATEGY TO SECURE OUR CITY’S 
FUTURE AGAINST THE CHALLENGES OF TODAY AND 
TOMORROW. WITH BOLD ACTIONS TO CONFRONT OUR 
CLIMATE CRISIS, ACHIEVE EQUITY, AND STRENGTHEN 
OUR DEMOCRACY, WE ARE BUILDING A STRONG AND 
FAIR CITY. JOIN US.
OneNYC 2050
BUILDING A STRONG AND FAIR CITY
MODERN
INFRA

In [ ]:
print(type(all_splits[0]))

In [ ]:
loader_seoul = PyPDFLoader("../data/2040_seoul_plan.pdf")
data_seoul = loader_seoul.load()
seoul_splits = text_splitter.split_documents(data_seoul)

print(len(seoul_splits))
for i, split in enumerate(seoul_splits):
    print(f"Split {i+1}:------------------------------------\n")
    print(split)

In [ ]:
print(seoul_splits[50].page_content)
print("-------------------------")
print(seoul_splits[51].page_content)

In [ ]:
for i in range(len(seoul_splits) - 1):
    seoul_splits[i].page_content += "\n" + seoul_splits[i + 1].page_content[:100]

print(seoul_splits[50].page_content)
print("---------------------------")
print(seoul_splits[51].page_content)

In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
import os
import time

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

embedding = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2", google_api_key=GEMINI_API_KEY
)
v = embedding.embed_query("뉴욕의 온실가스 저감 정책은 뭐야?")
print(len(v))

3072


In [16]:
from langchain_chroma import Chroma
import os
import time

persist_directory = "../chroma_store"

if not os.path.exists(persist_directory):
    print("Creating new Chroma store")
    # 429 Resource Exhausted 방지를 위해 청크 단위로 나누어 인덱싱을 수행합니다.
    chunk_size = 40
    # 첫 번째 청크로 vectorstore 초기화
    vectorstore = Chroma.from_documents(
        documents=seoul_splits[:chunk_size],
        embedding=embedding,
        persist_directory=persist_directory,
    )

    # 나머지 청크들을 10초 대기 시간을 두며 순차적으로 추가
    for i in range(chunk_size, len(seoul_splits), chunk_size):
        print(
            f"Adding documents from {i} to {min(i + chunk_size, len(seoul_splits))}..."
        )
        vectorstore.add_documents(seoul_splits[i : i + chunk_size])
        time.sleep(60)
    print("Chroma store creation completed!")
else:
    print("Loading existing Chroma store")
    vectorstore = Chroma(
        persist_directory=persist_directory, embedding_function=embedding
    )

Loading existing Chroma store


In [5]:
from langchain_chroma import Chroma
import os
import time

persist_directory = "../chroma_store"

chunk_size = 40

if os.path.exists(persist_directory):
    print("Loading existing Chroma store")
    vectorstore = Chroma(
        persist_directory=persist_directory, embedding_function=embedding
    )

    for i in range(0, len(all_splits), chunk_size):
        vectorstore.add_documents(all_splits[i : i + chunk_size])
        time.sleep(60)

Loading existing Chroma store


In [17]:
retriever = vectorstore.as_retriever(k=3)  # 청크 3개를 가져옴
docs = retriever.invoke("서울시의 환경 정책이 궁금해.")

for d in docs:
    print(d)
    print("------")

page_content='3.2 서울시 관련 실·국·본부 의견 167
‘호흡공동체’
 수도권 (미세먼지 등) 공동대응 상설협의체가 운영 중이며, 충남권까지 범위 확대의 필요성이 제기
되고 있음
 대기질과 관련하여 베이징 시와 공동 포럼을 개최하고 공동 연구를 진행하는 등 통합위원회 운영 중
2) 공원녹지기획팀
회의명 지속가능기반분과 실국면담 회의록(공원녹지기획팀)
날짜 2019.12.5.(목) 장소 서울시청 무교청사 9층
참석자 [공원녹지기획팀] 한정훈(공원녹지기획팀장), 김근주(공원녹지기획팀 주무관)
서울연구원 연구팀
❚ 회의 결과
장기적 지향점 및 정책 방향
 공원을 보존하고 공원 기능을 유지하도록 하는 것이 공원녹지정책과의 기본 방향이며, 이 부분을 담아 
공원녹지기본계획을 보완 중
- 공원녹지기본계획: 도시자연구역 지정 관련 내용을 업데이트하였고, 시의 의지에 따라 구역 내에서의 
행위에 대한 완화 내용과 함께 도시자연공원의 기능이 유지될 수 있도록 해제시키지 않도록 하는 
내용을 담음
 2030 서울플랜에서 주요하게 다룬 내용들을 공원녹지기본계획을 통해 시행하고 있으며, 2040 
서울도시기본계획에서 결정되는 내용에 맞추어서 추후 보완하고 방향성을 맞추어 진행할 계획
 미세먼지 저감과 기후변화 대응이라는 큰 목적을 위해 공원 및 녹지 확보가 중요
 시민 참여가 필요/중요하며, 녹지에 관한 의식을 고양하는데 효과적인 다양한 사업 진행 중
2040 서울도시기본계획에 바라는 점
 서울시정에서 생태 분야는 배정된 인력과 예산이 상대적으로 작은 분야임
 생물다양성에 관련한 생태 분야의 책무가 많으나 손을 대기가 힘들고, 장기미집행 등 정책우선순위가 
높은 사업에 밀려 시행이 어려움. 특히 산림과 수계를 벗어나 주택지 안에서도 생물다양성을 중시하는 
생활환경을 조성하는 것이 중요할 것으로 보임
 녹지축에 관련하여 생태경관보존지역을 확대하거나, 현재 지정된 지역을 체계적을 유지·관리 및 
보호하는 것이 필요함' metadata={'creationd

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 1. langchain_classic 패키지로부터 임포트하도록 변경
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_google_genai import ChatGoogleGenerativeAI

# 2. Chat 모델명을 gemini-1.5-flash로 수정
chat = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite", google_api_key=GEMINI_API_KEY
)

question_answering_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "사용자의 질문에 대해 아래 context에 기반하여 답변하라.:\n\n{context}",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)
document_chain = create_stuff_documents_chain(chat, question_answering_prompt)

In [19]:
from langchain_community.chat_message_histories import ChatMessageHistory

chat_history = ChatMessageHistory()
chat_history.add_user_message("서울시의 온실가스 저감 정책에 대해 알려줘.")

answer = document_chain.invoke(
    {
        "messages": chat_history.messages,
        "context": docs,
    }
)

chat_history.add_ai_message(answer)
print(answer)

제공해주신 자료에 따르면, 서울시의 온실가스 저감 및 환경 정책과 관련하여 환경정책과와 공원녹지기획팀이 제시한 주요 내용은 다음과 같습니다.

**1. 환경정책과 의견 (기후변화 대응 및 공간적 구현)**
*   **공간적 계획의 중요성:** 기후변화 대응과 에너지 효율화를 위해 도시기본계획 차원에서 공간적인 구현 방안을 마련해야 합니다.
*   **대기질 및 온실가스 관리:** 도시의 지형적 특성을 고려한 ‘바람길 조성’을 위해 고밀도 개발을 제한하는 등 대기질 관리와 온실가스 감축을 위한 공간적 계획이 필요합니다.
*   **자원순환 자립도시:** 폐기물 처리시설 및 자원회수시설과 관련하여 타 지역에 의존하지 않는 ‘자원순환 자립도시’를 지향합니다. 다만, 재활용 시설의 지하화는 주민 수용성 문제를 선결해야 한다고 보고 있습니다.
*   **통합적 접근:** 현재 개별적인 환경 종합계획과 에너지 종합계획이 추진되고 있으나, 이를 도시기본계획과 유기적으로 연결하여 반영하려는 노력이 강조되고 있습니다.

**2. 공원녹지기획팀 의견 (미세먼지 대응 및 생태적 접근)**
*   **미세먼지 및 기후변화 대응:** 미세먼지 저감과 기후변화 대응이라는 목적을 위해 공원 및 녹지를 확보하는 것이 핵심적인 정책 방향입니다.
*   **생물다양성 및 녹지축 관리:** 산림과 수계를 넘어 주택지 내에서도 생물다양성을 중시하는 생활환경을 조성하고, 생태경관보존지역 확대 및 체계적인 유지·관리가 필요합니다.
*   **광역적 협력:** 대기질 문제(미세먼지 등)와 관련하여 수도권 공동대응 상설협의체를 운영 중이며, 이를 충남권까지 확대하려는 움직임이 있습니다. 또한, 베이징시와 공동 연구 및 포럼을 진행하는 등 국제적인 협력을 수행하고 있습니다.

**3. 도시 비전과 방향성**
*   **대외적 시그널:** 미세먼지 문제 극복을 위해 도시 계획이 시민과 산업계에 명확한 비전을 전달해야 합니다. 뉴욕의 사례(PlaNYC, OneNYC)처럼 도시의 그린화와 생명, 물, 대기 문

In [ ]:
for m in chat_history.messages:
    print(m)

content='서울시의 온실가스 저감 정책에 대해 알려줘.' additional_kwargs={} response_metadata={}
content='제공해주신 자료에 따르면, 서울시의 온실가스 저감 및 환경 정책과 관련하여 환경정책과와 공원녹지기획팀이 제시한 주요 내용은 다음과 같습니다.\n\n**1. 환경정책과 의견 (기후변화 대응 및 공간적 구현)**\n*   **공간적 계획의 중요성:** 기후변화 대응과 에너지 효율화를 위해 도시기본계획 차원에서 공간적인 구현 방안을 마련해야 합니다.\n*   **대기질 및 온실가스 관리:** 도시의 지형적 특성을 고려한 ‘바람길 조성’을 위해 고밀도 개발을 제한하는 등 대기질 관리와 온실가스 감축을 위한 공간적 계획이 필요합니다.\n*   **자원순환 자립도시:** 폐기물 처리시설 및 자원회수시설과 관련하여 타 지역에 의존하지 않는 ‘자원순환 자립도시’를 지향합니다. 다만, 재활용 시설의 지하화는 주민 수용성 문제를 선결해야 한다고 보고 있습니다.\n*   **통합적 접근:** 현재 개별적인 환경 종합계획과 에너지 종합계획이 추진되고 있으나, 이를 도시기본계획과 유기적으로 연결하여 반영하려는 노력이 강조되고 있습니다.\n\n**2. 공원녹지기획팀 의견 (미세먼지 대응 및 생태적 접근)**\n*   **미세먼지 및 기후변화 대응:** 미세먼지 저감과 기후변화 대응이라는 목적을 위해 공원 및 녹지를 확보하는 것이 핵심적인 정책 방향입니다.\n*   **생물다양성 및 녹지축 관리:** 산림과 수계를 넘어 주택지 내에서도 생물다양성을 중시하는 생활환경을 조성하고, 생태경관보존지역 확대 및 체계적인 유지·관리가 필요합니다.\n*   **광역적 협력:** 대기질 문제(미세먼지 등)와 관련하여 수도권 공동대응 상설협의체를 운영 중이며, 이를 충남권까지 확대하려는 움직임이 있습니다. 또한, 베이징시와 공동 연구 및 포럼을 진행하는 등 국제적인 협력을 수행하고 있습니다.\n\n**3. 도시 비전과 방향성**\n*   **대외적 시

In [21]:
from langchain_core.output_parsers import StrOutputParser

query_for_nyc = "뉴욕은?"
# query augmentation
# 기존 대화 내용을 활용해 query_augmentation 수행
query_augmentation_prompt = ChatPromptTemplate.from_messages(
    [
        MessagesPlaceholder(variable_name="messages"),  # 기존 대화 내용
        (
            "system",
            "기존의 대화 내용을 활용하여 사용자가 질문한 의도를 파악해서 한 문장의 명료한 질문으로 변환하라. 대명사나 이, 저, 그와 같은 표현을 명확한 명사로 표현하라. :\n\n{query}",
        ),
    ]
)

In [22]:
query_agumentation_chain = query_augmentation_prompt | chat | StrOutputParser()

In [ ]:
agumented_query = query_agumentation_chain.invoke(
    {"messages": chat_history.messages, "query": query_for_nyc}
)

print(agumented_query)

 

---
*참고: 특정 기관의 공문서나 보고서 기반의 답변이므로, 실제 세부 사업 내용은 '서울시 기후환경본부' 또는 '2050 서울특별시 탄소중립 녹색성장 기본계획' 등을 통해 최신 업데이트를 확인하시는 것이 좋습니다.*


In [ ]:
docs = retriever.invoke(agumented_query)

for d in docs:
    print(d)
    print("------")

page_content='- 도시 기반시설 부문(건물): 무공해 빌딩 확대
- 도시 기반시설 부문(교통): 무공해 차량(ZEV) 보급 촉진
- 자원/산업 부문: 3R(줄이기, 재사용, 재활용), 플라스틱, 음식물쓰레기, HFC배출량 감소
- 기후변화 적응: 기후변화 적응 조치 강화
- 거버넌스: 기업(민간), 지자체, 주요 도시 등과 협력, 재생에너지 사업 투자 촉진
8) https://fpcj.jp/en/prlisting/tokyo_20211012/
9) https://zenbird.media/zero-emissions-tokyo-an-ambitious-climate-change-strategy/' metadata={'creator': 'PScript5.dll Version 5.2.2', 'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'page': 70, 'page_label': '71', 'total_pages': 272, 'source': '../data/2040_seoul_plan.pdf', 'title': '', 'moddate': '2023-02-14T18:21:42+09:00', 'author': '', 'creationdate': '2023-02-14T11:05:36+09:00'}
------
page_content='104 2. 시민참여와 미래상
부문 주요 계획과제 세부내용
교통
개인형 교통수단 활성화
전동 퀵 보드, 자전거 등 전용 도로 확보
자전거 도로 개선
공유 모빌리티 규제 완화
대중교통 편의 증진
쾌적하게 이용할 수 있는 대중교통 확충
심야버스 확충, 버스 총량제 폐지
고밀보다 충분한 교통 기반시설 마련
지하철역사 내 공간을 문화, 녹지 시설로 조성
보행친화도시 추구
골목 비우기 실천을 통한 총량 확보
보행친화도시로 개발
무 장 애  보 행 로  설 치
4대문 안 자가 차량 진입 금지
친환경 교통 정책 강화
CO2, 미세먼지 감소 위한 환경 규제
수소·전기차 충전시설 확충
차량 2부제 규제 강화

In [25]:
chat_history.add_user_message(query_for_nyc)

answer = document_chain.invoke({"messages": chat_history.messages, "context": docs})

chat_history.add_ai_message(answer)

print(answer)

제공해주신 자료 및 내용에 따르면, 뉴욕시는 도시가 직면한 환경 문제와 기후 변화에 대응하기 위해 **'PlaNYC'**와 **'OneNYC'**와 같은 포괄적인 미래 전략을 수립하여 실행하고 있습니다.

서울시의 정책 방향과 관련하여 언급된 뉴욕의 주요 전략적 특징은 다음과 같습니다:

*   **통합적 도시 비전 수립:** 뉴욕은 도시의 그린화(Green)를 핵심 의제로 삼아, 개별 사업 단위가 아닌 도시 전체의 비전을 담은 종합 계획(PlaNYC, OneNYC)을 수립했습니다. 이는 서울시가 향후 지향해야 할 융·복합적 대응 체계의 모델로 제시됩니다.
*   **환경 문제의 체계적 관리:** 단순한 탄소 저감을 넘어 **생명, 물, 대기 질 문제** 등을 포괄적으로 다룹니다. 도시의 물리적 환경을 개선하는 데 있어 이러한 요소들이 유기적으로 연결되도록 설계되었습니다.
*   **광역적 관계 설정:** 도시 내부의 정책에 그치지 않고, 인근 지역과의 관계를 설정하여 광역적인 차원에서 환경 문제에 접근하고 있습니다.
*   **시민 및 산업계와의 소통:** 환경 정책을 통해 도시가 나아가야 할 방향을 명확히 제시함으로써, 시민과 산업계에 강력한 대외적 시그널을 전달하고 협력을 유도하는 전략을 취하고 있습니다.

요약하자면, 뉴욕은 **'도시의 환경적 지속가능성'을 도시 발전의 핵심 동력으로 삼고, 이를 종합적인 공간 계획 및 전략(OneNYC 등)에 녹여내어 시민과 산업계가 함께 참여하는 통합적 관리 체계**를 구축하고 있는 사례로 평가됩니다.
